In [2]:
import HimalayanTokenization
tok = HimalayanTokenization.PyHimalayanTokenization(mode="LM")

In [3]:
print("=" * 70)
print("1. SEEDING VOCABULARY WITH AKSHARAS FROM TEST TEXT")
print("=" * 70)

# Sample texts to seed aksharas and to test
nepali_sample = "नेपाल एक सुन्दर देश हो। क्षत्रिय"
latin_sample = "Hello world 123"

# Use the DFA to harvest aksharas from the Nepali sample
# Note: harvest_aksharas normally reads a file, but we can just use dfa_tokenize_debug
aksharas = tok.dfa_tokenize_debug(nepali_sample)
# Also include each character in Latin sample? Not needed, Latin base is already seeded.

print(f"Harvested aksharas: {aksharas}")

# Initialize vocab with these aksharas
tok.initialize_vocab(
    aksharas=aksharas,
    seed_morphemes=[],      # no frozen morphemes
    punctuation=[".", ",", "।", "॥"],
    v_strict=[],            # no strict tokens
    v_ambiguous=[]          # no frequency‑gated tokens
)

print(f"Vocabulary size after initialisation: {tok.vocab_size()}")

print("\n" + "=" * 70)
print("2. ROUNDTRIP TEST")
print("=" * 70)

text = "नेपाल एक सुन्दर देश हो। Hello world! १२३"
encoded = tok.encode(text)
decoded = tok.decode(encoded)
print(f"Original:  {text}")
print(f"Decoded:   {decoded}")
print(f"Roundtrip: {'✅ PASS' if decoded == tok.normalize(text) else '❌ FAIL'}")

print("\n" + "=" * 70)
print("3. AKSHARA INTEGRITY (conjuncts)")
print("=" * 70)

conjunct_text = "क्ष"   # kṣa
tokens = tok.tokenize_to_strings(conjunct_text)
print(f"'{conjunct_text}' -> {tokens}")
if len(tokens) == 1 and tokens[0] == "क्ष":
    print("✅ Conjunct kept as one token")
else:
    print("❌ Conjunct split – DFA may not be active in encode path")

print("\n" + "=" * 70)
print("4. LATIN MARKER (should show ▂ before Latin words)")
print("=" * 70)

latin_tokens = tok.tokenize_to_strings(latin_sample)
print(f"'{latin_sample}' -> {latin_tokens}")
# Check if any token starts with ▂ (the Latin marker)
if any(t.startswith("▂") for t in latin_tokens):
    print("✅ Latin marker (▂) present")
else:
    print("❌ No Latin marker found – check marker logic")

print("\n" + "=" * 70)
print("5. MORPHOLOGY (optional – no paradigms loaded)")
print("=" * 70)

# With no paradigms, MorphScore defaults to 1.0 and root sets are empty.
# This is expected; you would need to add paradigms to test the constraint.
print("No paradigms added, so morphological filtering is inactive (root sets are empty).")
print("To test Morph, add a paradigm and call `assign_initial_roots()` and `enable_probabilistic_k()`.")

print("\n" + "=" * 70)
print("6. SAVE / LOAD VOCAB (optional)")
print("=" * 70)

# Optionally save the vocabulary (with the mode header)
# tok.save_vocab_tsv("test_vocab.tsv")
# tok.load_vocab_tsv("test_vocab.tsv")

print("\n✅ All tests completed successfully.")

1. SEEDING VOCABULARY WITH AKSHARAS FROM TEST TEXT
Harvested aksharas: ['ने', 'पा', 'ल', ' ', 'ए', 'क', ' ', 'सु', 'न्द', 'र', ' ', 'दे', 'श', ' ', 'हो', '।', ' ', 'क्ष', 'त्रि', 'य']
Vocabulary size after initialisation: 291

2. ROUNDTRIP TEST
Original:  नेपाल एक सुन्दर देश हो। Hello world! १२३
Decoded:   नेपाल एक सुन्दर देश हो। Hello world! १२३
Roundtrip: ✅ PASS

3. AKSHARA INTEGRITY (conjuncts)
'क्ष' -> ['▁', 'क्ष']
❌ Conjunct split – DFA may not be active in encode path

4. LATIN MARKER (should show ▂ before Latin words)
'Hello world 123' -> ['▂', 'H', 'e', 'l', 'l', 'o', '▂', 'w', 'o', 'r', 'l', 'd', '▂', '1', '2', '3']
✅ Latin marker (▂) present

5. MORPHOLOGY (optional – no paradigms loaded)
No paradigms added, so morphological filtering is inactive (root sets are empty).
To test Morph, add a paradigm and call `assign_initial_roots()` and `enable_probabilistic_k()`.

6. SAVE / LOAD VOCAB (optional)

✅ All tests completed successfully.


In [2]:
import pytest
import HimalayanTokenization as ht


VOCAB_PATH = "/home/lang-chain/Documents/HimalayanTokenizer/vocab_nepbpe/nepbpe_vocab_bilingual_v10.tsv"
SAMPLE_TEXT = "मेरो नाम राम हो।"


@pytest.fixture
def tokenizer():
    tok = ht.PyHimalyanTokenizer_Nepali(mode="LM")
    tok.load_vocab_tsv(VOCAB_PATH)
    return tok


def test_load_vocab_matching_mode(tokenizer):
    assert tokenizer is not None


def test_load_vocab_wrong_mode_raises():
    tok = ht.PyHimalyanTokenizer_Nepali(mode="segmentation")
    with pytest.raises(Exception):
        tok.load_vocab_tsv(VOCAB_PATH)


def test_encode_returns_ids(tokenizer):
    ids = tokenizer.encode(SAMPLE_TEXT)
    assert isinstance(ids, list)
    assert len(ids) > 0
    assert all(isinstance(i, int) for i in ids)


def test_decode_matches_normalized_input(tokenizer):
    ids = tokenizer.encode(SAMPLE_TEXT)
    decoded = tokenizer.decode(ids)
    assert decoded == tokenizer.normalize(SAMPLE_TEXT)


def test_verify_roundtrip(tokenizer):
    assert tokenizer.verify_roundtrip(SAMPLE_TEXT)


@pytest.mark.parametrize("text", [
    "मेरो नाम राम हो।",
    "नमस्ते संसार",
    "Hello नमस्ते mixed script",
    "",
])
def test_roundtrip_various_inputs(tokenizer, text):
    if text == "":
        pytest.skip("empty string edge case — confirm expected behavior first")
    assert tokenizer.verify_roundtrip(text)

In [5]:
import HimalayanTokenization as ht

VOCAB_PATH = "/home/lang-chain/Documents/HimalayanTokenizer/vocab_nepbpe/nepbpe_vocab_bilingual_v10.tsv"
TEXT = "मेरो नाम राम हो।"

def main():
    tok = ht.PyHimalayanTokenization(mode="LM")
    tok.load_vocab_tsv(VOCAB_PATH)

    ids = tok.encode(TEXT)
    decoded = tok.decode(ids)

    print(f"Input text     : {TEXT}")
    print(f"Token IDs      : {ids}")
    print(f"Number of tokens: {len(ids)}")
    print(f"Decoded text   : {decoded}")
    print(f"Normalized match: {decoded == tok.normalize(TEXT)}")
    print(f"Roundtrip OK   : {tok.verify_roundtrip(TEXT)}")

if __name__ == "__main__":
    main()

Input text     : मेरो नाम राम हो।
Token IDs      : [1866, 477, 4964, 477, 1840, 477, 1058, 100]
Number of tokens: 8
Decoded text   : मेरो नाम राम हो।
Normalized match: True
Roundtrip OK   : True


[load] warning: /home/lang-chain/Documents/HimalayanTokenizer/vocab_nepbpe/nepbpe_vocab_bilingual_v10.tsv has no mode header; assuming mode=LM


In [6]:
import sys
import statistics
import time

import HimalayanTokenization as ht
VOCAB_TSV = "/home/lang-chain/Documents/HimalayanTokenizer/vocab_nepbpe/nepbpe_vocab_bilingual_v10.tsv"

# MUST be identical to what you trained with (train.py). I4f these differ,
# normalization drifts and surface lookups miss.
FOLDING_RULES = [
    ("सङ्ग", "संग"),
    ("सँग", "संग"),
]

# 'Ġ' (U+0120) is the byte-alphabet surface for space (0x20). Without
# Ġ-prefixing, each inter-word space is its own token.
SPACE_PIECE = "\u2581" 


def show_piece(p: str) -> str:
    """Render a piece for display: space as ·, ZWNJ as <ZWNJ>."""
    if p == SPACE_PIECE:
        return "·"
    if p == "\u200c":
        return "<ZWNJ>"
    return p


SAMPLES = [
    "आज PyTorch मा transformer model train गरें।",
    "CUDA memory पर्याप्त नभएकाले batch size घटाएँ।",
    "LangChain प्रयोग गरेर RAG pipeline तयार गरियो।",
    "Vector database मा embeddings store गरियो।",
    "Model deployment Docker र Kubernetes मार्फत गरियो।",
    "Apache Spark प्रयोग गरेर data preprocessing गरियो।",
    "PySpark ले ५० लाख records process गर्यो।",
    "GitHub Actions प्रयोग गरेर CI/CD pipeline बनाइयो।",
    "MLflow मा experiment tracking गरियो।",
    "Tokenizer ले SentencePiece भन्दा कम tokens उत्पादन गर्यो।",
    "Hugging Face Transformers library प्रयोग गरियो।",
    "Fine-tuning गर्दा LoRA ले GPU memory बचायो।",
    "Azure OpenAI बाट inference call गरियो।",
    "Amazon Bedrock प्रयोग गरेर Claude model चलाइयो।",
    "TensorFlow भन्दा PyTorch debugging सजिलो लाग्यो।",
    "ChromaDB मा document embeddings index गरियो।",
    "FAISS प्रयोग गरेर semantic search तयार गरियो।",
    "FastAPI बाट inference API expose गरियो।",
    "Git repository मा Dockerfile update गरियो।",
    "OCR model ले नेपाली नागरिकताको जानकारी extract गर्यो।",
    "Transformer architecture ले translation quality सुधार गर्यो।",
    "Tokenizer को vocabulary size 64009 छ।",
    "Cross encoder reranker ले retrieval accuracy बढायो।",
    "Hybrid search मा BM25 र dense embeddings प्रयोग गरियो।",
    "Prompt engineering ले response quality सुधार गर्यो।"
]


def main(test_file=None) -> None:
    tok = ht.PyHimalayanTokenization(mode="LM")
    n = tok.load_vocab_tsv(VOCAB_TSV)
    print(f"loaded {n} tokens from {VOCAB_TSV}\n")

    raw_rates, content_rates = [], []
    tok_total = word_total = space_total = fails = 0

    print("=== sample tokenization ===")
    for idx, s in enumerate(SAMPLES, 1):
        print(f"  [{idx}/{len(SAMPLES)}] processing...", end="", flush=True)

        ids = tok.encode(s)
        pieces = [tok.get_token_surface(i) for i in ids]
        norm = tok.normalize(s)
        words = max(1, len(norm.split()))
        spaces = sum(1 for p in pieces if p == SPACE_PIECE)
        content = len(ids) - spaces
        ok = tok.decode(ids) == norm

        raw_rates.append(len(ids) / words)
        content_rates.append(content / words)
        tok_total += len(ids)
        word_total += words
        space_total += spaces
        if not ok:
            fails += 1

        shown = " ".join(show_piece(p) for p in pieces)
        print(f"\r  {s}")
        print(
            f"    {len(ids)} tok = {content} content + {spaces} space | "
            f"{len(ids)/words:.2f}/word ({content/words:.2f} ex-space) | "
            f"roundtrip={'OK' if ok else 'FAIL'}"
        )
        print(f"    {shown}")
        if not ok:
            print(f"    DECODED : {tok.decode(ids)!r}")
            print(f"    EXPECTED: {norm!r}")

    print("\n=== sample summary ===")
    print(
        f"  tokens/word   : mean={statistics.mean(raw_rates):.2f}  "
        f"median={statistics.median(raw_rates):.2f}"
    )
    print(
        f"  ex-space/word : mean={statistics.mean(content_rates):.2f}  "
        f"median={statistics.median(content_rates):.2f}   <- real subword fertility"
    )
    print(
        f"  micro/word    : {tok_total/max(1,word_total):.2f}  "
        f"(space tokens = {space_total}/{tok_total} = "
        f"{100*space_total/max(1,tok_total):.0f}%)"
    )
    print(f"  roundtrip     : {len(SAMPLES)-fails}/{len(SAMPLES)} OK")

    # Optional: fertility over a held-out file (fast, uses the Rust encode path).
    if test_file:
        print(f"\n=== fertility over {test_file} ===")
        space_id = tok.vocab_get_id(SPACE_PIECE)  # int (or None), computed once
        tt = ww = ss = lines = 0
        t0 = time.perf_counter()
        try:
            with open(test_file, encoding="utf-8") as f:
                for line_num, line in enumerate(f, 1):
                    line = line.strip()
                    if not line:
                        continue

                    if line_num % 1000 == 0:
                        elapsed = time.perf_counter() - t0
                        print(
                            f"  ... line {line_num}: {tt} tokens in {elapsed:.1f}s "
                            f"({tt/max(1,elapsed):.0f} tok/s)",
                            flush=True,
                        )

                    w = len(tok.normalize(line).split())
                    if w == 0:
                        continue

                    ids = tok.encode(line)
                    sp = ids.count(space_id) if space_id is not None else 0
                    tt += len(ids)
                    ss += sp
                    ww += w
                    lines += 1

                    if lines >= 20000:
                        print(f"  Reached {lines} lines limit", flush=True)
                        break

        except FileNotFoundError:
            print(f"  Error: File '{test_file}' not found. Skipping fertility analysis.")
            return
        except KeyboardInterrupt:
            print(f"\n  Interrupted after {lines} lines", flush=True)
            return

        dt = time.perf_counter() - t0
        print(
            f"  lines={lines} | tokens={tt} | tokens/word={tt/max(1,ww):.3f} | "
            f"ex-space/word={(tt-ss)/max(1,ww):.3f} | space-frac={ss/max(1,tt):.3f} | "
            f"{dt:.1f}s ({tt/max(1,dt):.0f} tok/s)"
        )


if __name__ == "__main__":
    try:
        if len(sys.argv) > 1 and not sys.argv[1].startswith("--f="):
            main(sys.argv[1])
        else:
            main()
    except KeyboardInterrupt:
        print("\nInterrupted by user", file=sys.stderr)

loaded 100001 tokens from /home/lang-chain/Documents/HimalayanTokenizer/vocab_nepbpe/nepbpe_vocab_bilingual_v10.tsv

=== sample tokenization ===
  आज PyTorch मा transformer model train गरें।
    12 tok = 11 content + 1 space | 1.71/word (1.57 ex-space) | roundtrip=OK
    ▁आज ▂Py Tor ch ▁मा ▂transformer ▂model ▂ train · गर ें।
  CUDA memory पर्याप्त नभएकाले batch size घटाएँ।
    12 tok = 11 content + 1 space | 1.71/word (1.57 ex-space) | roundtrip=OK
    ▂CUDA ▂memory ▁पर्याप्त ▁नभएकाले ▂batch ▂ size · घट ा एँ ।
  LangChain प्रयोग गरेर RAG pipeline तयार गरियो।
    12 tok = 10 content + 2 space | 1.71/word (1.43 ex-space) | roundtrip=OK
    ▂Lang Chain ▁प्रयोग ▁गरेर ▂RA G ▂pipeline · तयार · गरि यो।
  Vector database मा embeddings store गरियो।
    10 tok = 9 content + 1 space | 1.67/word (1.50 ex-space) | roundtrip=OK
    ▂Vector ▂database ▁मा ▂embedding s ▂ store · गरि यो।
  Model deployment Docker र Kubernetes मार्फत गरियो।
    12 tok = 10 content + 2 space | 1.71/word (1.43 ex-space) |

[load] warning: /home/lang-chain/Documents/HimalayanTokenizer/vocab_nepbpe/nepbpe_vocab_bilingual_v10.tsv has no mode header; assuming mode=LM


In [10]:
# demo.py
import HimalayanTokenization as ht

tok = ht.PyHimalayanTokenization(mode="LM")
tok.load_vocab_tsv("vocab.tsv")

text = "मेरो नाम राम हो।"
ids = tok.encode(text)
decoded = tok.decode(ids)

print(f"Input      : {text}")
print(f"Token IDs  : {ids}")
print(f"Num tokens : {len(ids)}")
print(f"Decoded    : {decoded}")
print(f"Roundtrip  : {tok.verify_roundtrip(text)}")

OSError: open vocab.tsv: No such file or directory (os error 2)

In [1]:
from HimalayanTokenization import PyHimalayanTokenization